In [ ]:
import json
import pandas as pd
import numpy as np

import cv2 
from typing import TypedDict

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Image, clear_output

import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.config.paths import *
from src.data.plots import plot_zero_boxes

ANALYSIS_DIR = CSV_DIR / "data_analysis"
IMAGE_DIR = IMAGE_DIR / "data_analysis"
PLOT_DIR = PLOT_DIR / "data_analysis"

In [ ]:
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def image_category(boxes):
    categories = {box["category"] for box in boxes}

    if categories == {0, 1}:
        return "both"
    elif categories == {1}:
        return "fire_only"
    elif categories == {0}:
        return "smoke_only"
    else:
        return "neither"

In [ ]:
df = pd.read_csv(CSV_DIR/"data/metadata.csv")

df["boxes"] = df["boxes"].apply(json.loads)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,6))

fire_split = df["label"].map({0: "Non-fire/smoke", 1: "Fire/Smoke"}).value_counts()
labels = [f"{label} ({count:,})" for label, count in fire_split.items()]

fire_split.plot(
    kind="pie",
    autopct="%1.1f%%",
    labels=labels,
    ax=axes[0]
)

df["category"] = df["boxes"].apply(image_category)
cat_counts = df.loc[
    df["category"] != "neither", # Exclude neither because its already shown in fire/non-fire
    "category"
].value_counts()
labels = [f"{label} ({count})" for label, count in cat_counts.items()]

cat_counts.plot(
    kind="pie",
    autopct="%1.1f%%",
    labels=labels,
    ax=axes[1]
)

plt.tight_layout()
fig.savefig(PLOT_DIR / "image_category_distribution.svg")
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))

split_counts = df["split"].value_counts()
labels = [f"{label} ({count:,})" for label, count in split_counts.items()]

split_counts.plot(
    kind="pie",
    autopct="%1.1f%%",
    labels=labels,
)

plt.tight_layout()
plt.savefig(PLOT_DIR / "data_split.svg")
plt.show()

In [ ]:
plt.figure(figsize=(18, 6))
order = ["neither", "fire_only", "smoke_only", "both"]
ax = (df.groupby("split")["category"].value_counts(normalize=True).unstack()
      .reindex(columns=order)
      .reindex(["train", "val", "test"])
      .plot(kind="bar", stacked=True)
      )

ax.legend(title="Category", labels=["Neither", "Fire Only", "Smoke Only", "Both"], bbox_to_anchor=(1.3, 0.2), loc="lower right")

split_sizes = df.groupby("split").size().reindex(["train", "val", "test"])

for c in ax.containers:
    labels = [
        f"{round(v.get_height() * split_sizes.iloc[i]):,}"
        if v.get_height() > 0 else ""
        for i, v in enumerate(c)
    ]

    ax.bar_label(c, labels=labels, label_type="center")

plt.title("Category Distribution by Split")

plt.xlabel("Split")
plt.ylabel("Proportion")
plt.yticks(np.arange(0, 1.1, 0.1))

plt.savefig(PLOT_DIR / "category_distribution_by_split.svg")
plt.show()

Fire looks a lot smaller than smoke however this is only counting one instance of each which may not reflect how many examples there is actually in the images as some images have multi-class of both fire and smoke so there could possibly be more fire examples than smoke

In [ ]:
sizes_df = df["boxes"].explode().dropna().apply(pd.Series)

sizes_df[["xc", "yc", "width", "height"]] = pd.DataFrame(
    sizes_df["coordinates"].tolist(),
    index=sizes_df.index
)

sizes_df["path"] = df.loc[sizes_df.index, "path"].to_numpy()
sizes_df["area"] = sizes_df["width"] * sizes_df["height"]
sizes_df["split"] = df.loc[sizes_df.index, "split"].to_numpy()

sizes_df.to_csv(ANALYSIS_DIR / "box_sizes.csv", index=False)
sizes_df

In [ ]:
sizes_summary = sizes_df.groupby("category")["area"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)

sizes_summary

In [ ]:
fire_count = sizes_summary["count"][1]
smoke_count = sizes_summary["count"][0]

labels = [f"Fire ({fire_count})", f"Smoke ({smoke_count})"]

plt.figure(figsize=(10, 8))

plt.pie(
    [fire_count, smoke_count],
    autopct="%1.1f%%",
    labels=labels
)

plt.savefig(PLOT_DIR / "label_distribution.svg")
plt.show()

In [ ]:
# Uses metadata_raw privately to plot these doesnt use current metadata
image_paths = plot_zero_boxes()
current = 0

prev_btn = widgets.Button(description="Previous")
next_btn = widgets.Button(description="Next")
output = widgets.Output()

def show_image():
    with output:
        clear_output(wait=True)
        display(Image(filename=str(image_paths[current])))

    prev_btn.disabled = current == 0
    next_btn.disabled = current == len(image_paths) - 1

def navigate(change):
    global current

    if change == next_btn:
        current += 1
    else:
        current -= 1

    show_image()

prev_btn.on_click(navigate)
next_btn.on_click(navigate)

display(widgets.HBox([prev_btn, next_btn]), output)
show_image()

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))

sizes_df.boxplot(
    column="area",
    by="category",
    ax=ax,
    vert=False,
)

plt.savefig(PLOT_DIR / "box_plot_fire_vs_smoke_area.svg")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))

sizes_df.boxplot(
    column="area",
    ax=ax,
    vert=False
)

plt.savefig(PLOT_DIR / "box_plot_areas.svg")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 10))

for ax, split in zip(axes, ["train", "val", "test"]):
    ax.boxplot(
        [
            sizes_df["area"][(sizes_df["split"] == split) & (sizes_df["category"] == 1)],
            sizes_df["area"][(sizes_df["split"] == split) & (sizes_df["category"] == 0)],
            sizes_df["area"][sizes_df["split"] == split],
        ],
        orientation="horizontal",
        tick_labels=["Fire", "Smoke", "All boxes"],
        showfliers=False
    )
    ax.set_xticks(np.arange(0, 1.1, 0.1))
    ax.grid(axis="x", linestyle="--", alpha=0.5)
    ax.set_title(split)

plt.savefig(PLOT_DIR / "area_dist_by_split.svg")
plt.show()

In [ ]:
class ImageProperties(TypedDict):
    width: int
    height: int

    brightness: float
    sharpness: float
    contrast: float

    split: str
    category: str
    path: str
    

In [ ]:
stats = []

for row in df.itertuples():
    img = cv2.imread(str(DATASET_DIR / row.path), cv2.IMREAD_GRAYSCALE)

    h, w = img.shape

    stats.append(ImageProperties(
        width=w,
        height=h,
        brightness=img.mean(),
        sharpness=cv2.Laplacian(img, cv2.CV_64F).var(),
        contrast=img.std(),
        split=row.split,
        category=row.category,
        path=row.path
    ))

stats = pd.DataFrame(stats)
stats.to_csv(ANALYSIS_DIR/"image_properties.csv", index=False)

stats

In [ ]:
plt.figure(figsize=(10, 5))

plt.scatter(
    stats["width"],
    stats["height"],
)

plt.xlabel("Width")
plt.ylabel("Height")

plt.show()

#TODO find a way to plot the 2 trendlines i can see in this data of the aspect ratios

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 12))

for ax, prop in zip(axes, ["brightness", "contrast", "sharpness"]):
    ax.boxplot(
        [
            stats[prop][stats["category"] == "neither"],
            stats[prop][stats["category"] == "both"],
            stats[prop][stats["category"] == "smoke_only"],
            stats[prop][stats["category"] == "fire_only"],
        ],
        tick_labels=["Neither", "Both", "Smoke", "Fire"]
    )
    ax.set_title(prop)

plt.savefig(PLOT_DIR / "image_prop_box_plots.svg")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 12))

for ax, prop in zip(axes, ["brightness", "contrast", "sharpness"]):
    ax.boxplot(
        [
            stats[prop][stats["split"] == split]          
            for split in ["train", "val", "test"]
        ],
        tick_labels=["Train", "Val", "Test"],
        showfliers=False
    )
    ax.set_title(prop)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.savefig(PLOT_DIR / "image_props_by_split.svg")
plt.show()
